In [7]:
!pip install qiskit==1.4.3
!pip install qiskit_nature==0.7.2
!pip install qiskit_aer==0.17.1
!pip install pyscf
!pip install rdkit

  Using cached qiskit-1.4.3-cp39-abi3-macosx_11_0_arm64.whl.metadata (12 kB)
Using cached qiskit-1.4.3-cp39-abi3-macosx_11_0_arm64.whl (6.3 MB)
  Attempting uninstall: qiskit
    Found existing installation: qiskit 2.3.0
    Uninstalling qiskit-2.3.0:
      Successfully uninstalled qiskit-2.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
qiskit-ibm-runtime 0.45.1 requires qiskit>=2.0.0, but you have qiskit 1.4.3 which is incompatible.


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

def cap_structure(input_pdb, output_pdb):
    """
    Reads a PDB file, identifies and caps dangling bonds with hydrogen atoms,
    and saves the capped structure to a new PDB file.
    """
    # Read the molecule from the PDB file, sanitizing it to fix valency issues
    mol = Chem.MolFromPDBFile(input_pdb, sanitize=True, removeHs=False)

    if mol is None:
        print(f"Error: Could not read molecule from {input_pdb}")
        return

    # Add hydrogens to all atoms to ensure full valency
    mol = Chem.AddHs(mol, explicitOnly=True)

    # Save the new molecule with the added hydrogens
    writer = Chem.PDBWriter(output_pdb)
    writer.write(mol)
    writer.close()

    print(f"Structure from '{input_pdb}' has been capped and saved to '{output_pdb}'")

if __name__ == "__main__":
    input_filea = "res_17_21_capped.pdb"
    output_filea = "res_17_21_cappeda.pdb"
    cap_structure(input_filea, output_filea)
    input_filew = "wildtype.pdb"
    output_filew = "res_17_21_cappedw.pdb"
    cap_structure(input_filew, output_filew)

In [ ]:
def convert_pdb_to_xyz(input_pdb, output_xyz):
    """
    Converts a PDB file to an XYZ file using RDKit.
    """
    # Read the molecule from the PDB file
    mol = Chem.MolFromPDBFile(input_pdb, sanitize=False, removeHs=False)

    if mol is None:
        print(f"Error: Could not read molecule from {input_pdb}")
        return

    # Create an XYZ file writer
    writer = Chem.PDBWriter(output_xyz)
    writer.write(mol)
    writer.close()


    with open(output_xyz, 'w') as f:
        # Write the number of atoms
        f.write(f"{mol.GetNumAtoms()}\n\n")
        # Write each atom's symbol and coordinates
        for atom in mol.GetAtoms():
            pos = mol.GetConformer().GetAtomPosition(atom.GetIdx())
            symbol = atom.GetSymbol()
            f.write(f"{symbol}\t{pos.x:.4f}\t{pos.y:.4f}\t{pos.z:.4f}\n")

    print(f"Successfully converted '{input_pdb}' to '{output_xyz}'")

if __name__ == "__main__":
    input_filea = "res_17_21_cappeda.pdb"
    output_filea = "res_17_21_cappeda.xyz"
    convert_pdb_to_xyz(input_filea, output_filea)
    input_filew = "res_17_21_cappedw.pdb"
    output_filew = "res_17_21_cappedw.xyz"
    convert_pdb_to_xyz(input_filew, output_filew)

In [13]:
import numpy as np
#import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B, SPSA, SLSQP
from qiskit_nature.second_q.transformers import FreezeCoreTransformer, ActiveSpaceTransformer
from qiskit_nature.second_q.formats.molecule_info import MoleculeInfo as Molecule
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.units import DistanceUnit
from qiskit.circuit.library import TwoLocal
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.drivers import PySCFDriver, MethodType
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit.circuit.library import EfficientSU2
from qiskit_aer import AerSimulator, Aer
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import EstimatorV2, Estimator
import qiskit_nature.settings
from pyscf import solvent, gto, scf
from pyscf.solvent import ddCOSMO

qiskit_nature.settings.use_pauli_sum_op = False

ImportError: cannot import name 'BaseEstimator' from 'qiskit.primitives' (/Users/kushalpatil/Desktop/JavaSwerve2026/.venv/lib/python3.12/site-packages/qiskit/primitives/__init__.py)

In [ ]:
import qiskit.utils
import qiskit_nature.utils
import qiskit_aer.utils

print(f"qiskit version: {qiskit.__version__}")
print(f"qiskit-nature version: {qiskit_nature.__version__}")
print(f"qiskit-aer version: {qiskit_aer.__version__}")

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver, MethodType
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import ParityMapper, JordanWignerMapper
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from pyscf import gto, scf, solvent
from pyscf.tools import cubegen

def log_orbital_energies(name, orbital_energies, orbital_occupations, log_list, orbitals=[2,3,4,5]):
    for i in orbitals:
        log_list.append({
            "fragment": name,
            "orbital_index": i,
            "energy": orbital_energies[i],
            "occupation": orbital_occupations[i]
        })
    print(log_list)
    return log_list

def create_abeta_driver(atom_string, charge, spin, name, basis='sto-3g'):
    """
    Creates and returns a Qiskit-compatible ElectronicStructureProblem.
    Solvent effects (ddCOSMO) are applied manually in PySCF but not captured in Qiskit integrals.
    """
    # Manual PySCF run with ddCOSMO (for reference or logging)
    mol = gto.Mole()
    mol.atom = atom_string
    mol.unit = 'Angstrom'
    mol.basis = basis
    mol.charge = charge
    mol.spin = spin
    mol.build()

    mf = scf.RHF(mol).ddCOSMO()
    mf.with_solvent.eps = 80.0
    mf.kernel(level_shift=0.2)
    print("Total RHF energy:", mf.e_tot, "Hartree")
    mo_coeff1 = mf.mo_coeff.copy()
    mo_occ1 = mf.mo_occ.copy()
    
    orbital1s = [150, 151, 152, 153]
    for i in orbital1s:
        cubegen.orbital(mol, f'orbital{i}_{name}.cube', mo_coeff1[:,i])


    driver = PySCFDriver(
        atom=atom_string,
        unit=DistanceUnit.ANGSTROM,
        basis=basis,
        charge=charge,
        spin=spin,
        method=MethodType.RHF
    )
    problem = driver.run()

    return problem

def get_homo_lumo_orbitals(problem, window=2):
    occ = problem.orbital_occupations
    homo_index = max(i for i, occ_val in enumerate(occ) if occ_val > 0)
    lumo_index = homo_index + 1
    return list(range(homo_index - window + 1, lumo_index + window))

def get_qubit_op(xyz_file_content, name):
    # Parse XYZ content
    lines = xyz_file_content.strip().split('\n')
    atom_data_string = '\n'.join(lines[2:]) if len(lines) > 2 else xyz_file_content.strip()

    current_charge = 0
    current_spin = 1

    # Step 1: Create initial problem
    problem = create_abeta_driver(atom_data_string, current_charge, current_spin, name)
    problem = create_abeta_driver(atom_data_string, current_charge, current_spin, name)

    for i, (occ, e) in enumerate(zip(problem.orbital_occupations, problem.orbital_energies)):
        print(f"[{name}] Orbital {i}: occ={occ:.2f}, energy={e:.6f} Hartree")

    orbital_log = []

    log_orbital_energies(name, problem.orbital_energies, problem.orbital_occupations, orbital_log)


    print(f"[{name}] Initial num_particles: {problem.num_particles}")
    print(f"[{name}] Initial num_spatial_orbitals: {problem.num_spatial_orbitals}")
    ansatz = TwoLocal(rotation_blocks='ry', entanglement_blocks='cz', reps=1)
    vqe_solver = VQE(EstimatorV2(), ansatz, SLSQP())
    solver = GroundStateEigensolver(JordanWignerMapper(), vqe_solver)
    # Step 2: Determine active orbitals
    if name == "wt":
       # Step 2: Determine active orbitals
        #temp_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=4, active_orbitals=list(range(4)))
        #temp_problem = temp_transformer.transform(problem)
        #temp_result = solver.solve(temp_problem)
        #active_orbitals = get_homo_lumo_orbitals(temp_problem, window=2)

        active_orbitals = [150, 151, 152, 153] #wt hardcoded
    else:
       # Step 2: Determine active orbitals
        #temp_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=4, active_orbitals=list(range(4)))
        #temp_problem = temp_transformer.transform(problem)
        #temp_result = solver.solve(temp_problem)
        #active_orbitals = get_homo_lumo_orbitals(temp_problem, window=2)
       active_orbitals = [150, 151, 152, 153]  # Arctic mutant hardcoded

    print(f"[{name}] Active orbitals selected: {active_orbitals}")

    # Step 3: Apply final transformer
    transformer = ActiveSpaceTransformer(
        num_electrons=4,
        num_spatial_orbitals=4,
        active_orbitals=active_orbitals
    )
    problem = transformer.transform(problem)

    # Step 4: Map to qubit Hamiltonian
    num_particles = problem.num_particles
    num_spatial_orbitals_transformed = problem.num_spatial_orbitals
    print(f"[{name}] Transformed num_particles: {num_particles}")
    print(f"[{name}] Transformed num_spatial_orbitals: {num_spatial_orbitals_transformed}")

    mapper = JordanWignerMapper()
    hamiltonian = mapper.map(problem.second_q_ops()[0])
    print(f"[{name}] Hamiltonian qubits: {hamiltonian.num_qubits}")

    return hamiltonian, num_particles, num_spatial_orbitals_transformed, problem, mapper


In [3]:
def read_xyz_file(filepath, charge, mult):
    with open(filepath, 'r') as f:
        lines = f.readlines()[2:]  # Skip atom count and comment
    mol_lines = [f"{charge} {mult}"]
    for line in lines:
        parts = line.split()
        if len(parts) < 4:
            continue
        atom = parts[0]
        x, y, z = map(float, parts[1:4])
        mol_lines.append(f"{atom} {x:.6f} {y:.6f} {z:.6f}")
    return "\n".join(mol_lines)

wt_xyz_content = read_xyz_file("res_17_21_cappedw.xyz", 0, 1)
arctic_xyz_content = read_xyz_file("res_17_21_cappeda.xyz", 0, 1)
print(wt_xyz_content)
print(f"Arctic: {arctic_xyz_content}")

0 1
N 44.330000 22.840000 53.410000
H 43.970000 22.400000 54.250000
C 43.390000 23.510000 52.530000
H 43.170000 22.710000 51.820000
C 42.180000 24.010000 53.300000
H 42.550000 24.730000 54.030000
H 41.760000 23.140000 53.810000
C 40.980000 24.610000 52.560000
H 41.320000 25.560000 52.160000
C 40.530000 23.720000 51.400000
H 41.350000 23.700000 50.680000
H 40.290000 22.700000 51.710000
H 39.630000 24.180000 50.980000
C 39.860000 24.900000 53.540000
H 39.610000 23.970000 54.050000
H 40.170000 25.550000 54.360000
H 38.920000 25.310000 53.160000
C 43.950000 24.660000 51.700000
O 43.850000 24.670000 50.470000
N 44.580000 25.650000 52.330000
H 44.490000 25.640000 53.340000
C 45.070000 26.800000 51.600000
H 44.140000 27.210000 51.210000
C 45.460000 27.890000 52.580000
H 44.730000 27.860000 53.390000
C 46.840000 27.670000 53.200000
H 46.730000 26.700000 53.670000
H 47.650000 27.640000 52.470000
H 47.020000 28.540000 53.830000
C 45.100000 29.270000 52.020000
H 45.790000 29.560000 51.220000
H 44

In [ ]:
wt_hamiltonian, wt_particles, wt_orbitals, wt_problem, wt_mapper = get_qubit_op(wt_xyz_content, name="wt")
arctic_hamiltonian, arctic_particles, arctic_orbitals, arctic_problem, arctic_mapper = get_qubit_op(arctic_xyz_content, name="mut")

def run_vqe_with_plot(hamiltonian, problem, mapper, num_particles, num_orbitals, label=""):
    # Initialize lists to store convergence data
    energy_values = []
    iteration_counts = []
    
    # Callback function to track convergence
    def callback(eval_count, parameters, mean, std):
        energy_values.append(mean)
        iteration_counts.append(eval_count)
        print(f"{label} - Iteration: {eval_count}, Energy: {mean}")
    
    # Set up ansatz 
    num_qubits = hamiltonian.num_qubits
    init_state = HartreeFock(num_orbitals, num_particles, mapper)
    ansatz_body = TwoLocal(
        num_qubits=num_qubits,
        rotation_blocks=['ry', 'rz'],
        entanglement_blocks='cx',
        entanglement='linear',
        reps=2,
        insert_barriers=True,
    )
    ansatz = QuantumCircuit(num_qubits)
    ansatz.append(init_state, list(range(num_qubits)))
    ansatz.append(ansatz_body, list(range(num_qubits)))
    ansatz.draw(output='text', fold=80, plot_barriers=False, reverse_bits=True)
    print(ansatz.draw())
    
    # Run VQE with callback
    vqe = VQE(
        estimator=Estimator(),
        ansatz=ansatz,
        optimizer=SLSQP(maxiter=100),
        callback=callback
    )
    result = vqe.compute_minimum_eigenvalue(hamiltonian)
    interpreted_result = problem.interpret(result)
    
    # Return both the result and convergence data
    return {
        'result': interpreted_result,
        'energies': energy_values,
        'iterations': iteration_counts,
        'final_energy': interpreted_result.total_energies[0]
    }

# Run VQE for both systems
wt_data = run_vqe_with_plot(wt_hamiltonian, wt_problem, wt_mapper, wt_particles, wt_orbitals, label="WT")
arctic_data = run_vqe_with_plot(arctic_hamiltonian, arctic_problem, arctic_mapper, arctic_particles, arctic_orbitals, label="Arctic")

# Create convergence plot
plt.figure(figsize=(10, 6))
plt.plot(wt_data['iterations'], wt_data['energies'], 'b-', label='WT', linewidth=2)
plt.plot(arctic_data['iterations'], arctic_data['energies'], 'r-', label='Arctic', linewidth=2)

# Add horizontal lines for final energies
#plt.axhline(y=wt_data['final_energy'], color='b', linestyle='--', alpha=0.5)
#plt.axhline(y=arctic_data['final_energy'], color='r', linestyle='--', alpha=0.5)

# Format plot
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Expectation Energy (hartree)', fontsize=12)
plt.title('VQE Convergence: WT vs Arctic', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()


plt.savefig('vqe_convergence_comparison1.png', dpi=300)
plt.show()

# Compare final energies
print(f"\nFinal Energies:")
print(f"WT:     {wt_data['final_energy']} hartree")
print(f"Arctic: {arctic_data['final_energy']} hartree")
print(f"ΔE:     {arctic_data['final_energy'] - wt_data['final_energy']} hartree")

In [ ]:
import mdtraj as md

traj1 = md.load_pdb("res_17_21_cappedw.pdb")
traj2 = md.load_pdb("res_17_21_cappeda.pdb")

backbone_atoms = ['N', 'CA', 'C']
backbone_indices = [atom.index for atom in traj1.topology.atoms if atom.name in backbone_atoms]

rms = md.rmsd(traj2, traj1, atom_indices=backbone_indices)
print(rms)

In [ ]:
import matplotlib.pyplot as plt

fragments = ['WT', 'Arctic']
energies = [-1788.3671, -1789.4439]  # Hartree

plt.figure(figsize=(6, 4))
plt.bar(fragments, energies, color=['steelblue', 'darkred'])
plt.ylabel('VQE Energy (Hartree)')
plt.title('VQE Ground State Energy: WT vs Arctic')
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [16]:
wt_co="""N 44.330000 22.840000 53.410000
H 43.970000 22.400000 54.250000
C 43.390000 23.510000 52.530000
H 43.170000 22.710000 51.820000
C 42.180000 24.010000 53.300000
H 42.550000 24.730000 54.030000
H 41.760000 23.140000 53.810000
C 40.980000 24.610000 52.560000
H 41.320000 25.560000 52.160000
C 40.530000 23.720000 51.400000
H 41.350000 23.700000 50.680000
H 40.290000 22.700000 51.710000
H 39.630000 24.180000 50.980000
C 39.860000 24.900000 53.540000
H 39.610000 23.970000 54.050000
H 40.170000 25.550000 54.360000
H 38.920000 25.310000 53.160000
C 43.950000 24.660000 51.700000
O 43.850000 24.670000 50.470000
N 44.580000 25.650000 52.330000
H 44.490000 25.640000 53.340000
C 45.070000 26.800000 51.600000
H 44.140000 27.210000 51.210000
C 45.460000 27.890000 52.580000
H 44.730000 27.860000 53.390000
C 46.840000 27.670000 53.200000
H 46.730000 26.700000 53.670000
H 47.650000 27.640000 52.470000
H 47.020000 28.540000 53.830000
C 45.100000 29.270000 52.020000
H 45.790000 29.560000 51.220000
H 44.060000 29.300000 51.710000
H 45.200000 30.040000 52.780000
C 46.040000 26.530000 50.450000
O 46.030000 27.340000 49.520000
N 46.820000 25.450000 50.500000
H 46.870000 24.910000 51.360000
C 47.730000 25.000000 49.470000
H 48.350000 25.850000 49.160000
C 48.640000 23.910000 50.040000
H 48.040000 23.230000 50.640000
H 49.350000 24.430000 50.680000
C 49.360000 23.040000 49.040000
C 50.540000 23.500000 48.460000
H 50.960000 24.450000 48.770000
C 51.190000 22.750000 47.470000
H 52.150000 23.080000 47.100000
C 50.670000 21.510000 47.070000
H 51.140000 20.880000 46.330000
C 49.480000 21.060000 47.640000
H 49.080000 20.120000 47.310000
C 48.830000 21.810000 48.640000
H 47.910000 21.400000 49.040000
C 46.970000 24.630000 48.210000
O 47.430000 24.940000 47.120000
N 45.900000 23.840000 48.340000
H 45.660000 23.610000 49.290000
C 45.090000 23.230000 47.300000
H 45.800000 22.840000 46.570000
C 44.250000 22.080000 47.870000
H 43.480000 21.800000 47.150000
H 43.860000 22.560000 48.770000
C 45.130000 20.910000 48.250000
C 45.680000 20.090000 47.260000
H 45.730000 20.390000 46.230000
C 46.320000 18.870000 47.520000
H 46.700000 18.250000 46.720000
C 46.490000 18.520000 48.860000
H 47.090000 17.670000 49.150000
C 45.870000 19.300000 49.850000
H 45.750000 18.910000 50.850000
C 45.140000 20.460000 49.570000
H 44.600000 20.940000 50.380000
C 44.250000 24.360000 46.720000
O 44.260000 24.720000 45.550000
N 43.510000 25.090000 47.560000
H 43.410000 24.750000 48.510000
C 42.580000 26.100000 47.100000
H 41.890000 25.610000 46.410000
C 41.950000 26.660000 48.370000
H 41.160000 27.340000 48.050000
H 41.600000 25.790000 48.920000
H 42.620000 27.140000 49.090000
C 43.270000 27.320000 46.500000
O 42.780000 27.820000 45.490000
"""

arctic_cordinates="""N 57.575000 55.467000 56.497000
H 56.625000 55.777000 56.307000
C 58.685000 56.207000 55.827000
H 59.355000 56.637000 56.577000
C 59.565000 55.257000 55.007000
H 59.865000 54.437000 55.657000
H 60.475000 55.797000 54.767000
C 58.985000 54.717000 53.677000
H 58.655000 55.497000 52.997000
C 60.122000 54.030000 52.867000
H 59.802000 53.490000 51.977000
H 60.832000 54.780000 52.537000
H 60.672000 53.320000 53.487000
C 57.875000 53.630000 53.927000
H 58.175000 52.730000 54.457000
H 57.025000 54.060000 54.437000
H 57.535000 53.280000 52.947000
C 58.135000 57.387000 54.987000
O 56.995000 57.397000 54.627000
N 58.995000 58.317000 54.667000
H 59.915000 58.227000 55.087000
C 58.885000 59.287000 53.557000
H 57.985000 59.887000 53.737000
C 59.915000 60.417000 53.567000
H 60.155000 60.667000 54.597000
C 61.202000 59.977000 52.897000
H 61.992000 60.687000 53.137000
H 61.592000 59.007000 53.217000
H 61.072000 59.897000 51.827000
C 59.425000 61.767000 53.037000
H 59.185000 61.737000 51.977000
H 58.545000 62.057000 53.617000
H 60.225000 62.477000 53.257000
C 58.755000 58.607000 52.177000
O 59.255000 57.507000 51.837000
N 57.995000 59.337000 51.387000
H 57.745000 60.287000 51.617000
C 57.895000 59.147000 49.967000
H 58.735000 58.537000 49.637000
C 56.485000 58.647000 49.507000
H 55.645000 59.267000 49.837000
H 56.315000 57.727000 50.057000
C 56.265000 58.577000 47.977000
C 55.345000 59.507000 47.417000
H 54.705000 60.137000 48.017000
C 55.275000 59.617000 46.037000
H 54.655000 60.397000 45.607000
C 55.985000 58.707000 45.247000
H 55.905000 58.737000 44.167000
C 56.845000 57.797000 45.797000
H 57.415000 57.137000 45.157000
C 57.035000 57.707000 47.177000
H 57.745000 57.037000 47.637000
C 58.075000 60.427000 49.177000
O 57.525000 61.507000 49.577000
N 58.705000 60.317000 48.027000
H 59.175000 59.437000 47.877000
C 58.785000 61.237000 46.927000
H 57.805000 61.367000 46.467000
C 59.395000 62.587000 47.407000
H 60.315000 62.597000 47.987000
H 58.645000 63.107000 47.987000
C 59.775000 63.527000 46.287000
C 58.765000 64.127000 45.527000
H 57.755000 64.007000 45.877000
C 59.155000 64.877000 44.407000
H 58.345000 65.287000 43.827000
C 60.512000 65.047000 44.057000
H 60.732000 65.767000 43.287000
C 61.532000 64.587000 44.947000
H 62.572000 64.787000 44.777000
C 61.162000 63.727000 45.997000
H 61.922000 63.307000 46.637000
C 59.535000 60.607000 45.767000
O 59.095000 60.697000 44.617000
N 60.752000 60.157000 45.957000
H 61.122000 60.087000 46.887000
C 61.432000 59.337000 44.937000
H 61.382000 59.917000 44.017000
C 62.912000 59.247000 45.427000
H 63.612000 58.997000 44.637000
H 63.212000 60.247000 45.747000
H 63.112000 58.517000 46.207000
C 60.882000 57.947000 44.637000
O 60.035000 57.447000 45.387000"""